# MetaAlgorithmGA Testing - Dynamic Algorithm Selection

Test GA optimization with selectable algorithms using the NEW MetaConfig system.

## 1. Imports & Setup

In [1]:
import time
import numpy as np
import matplotlib.pyplot as plt

from tests.fixtures.graphs import _create_clustered_graph

from src.meta.config.meta_config import MetaConfig, GAConfig, Algorithms
from src.meta.core.algorithm_registry import AlgorithmRegistry
from src.meta.parameterizers.algorithm_parameterizer import UnifiedAlgorithmParameterizer

print('✓ All imports loaded successfully')

✓ All imports loaded successfully


## 2. Helper Functions

In [2]:
# Import all helper functions from the helpers module
from notebooks.helpers import (
    fixture_to_graph,
    format_time,
    get_optimal_weight,
    get_baseline_fitness,
    get_cascading_baseline,
    get_individual_algorithm_weights,
    run_ga_evaluation,
)

print('✓ Helper functions imported from notebooks.helpers (clean notebook, no duplication)')

✓ Helper functions imported from notebooks.helpers (clean notebook, no duplication)


## 3. Configuration - Choose Algorithms to Test

In [13]:
# Graph parameters
NR_OF_NODES = 1000
SEEDS = [43, 73, 98]

# Execution Configs
POPULATION_SIZE = 20
GENERATIONS = 10
MUTATION_RATE = 0.25

# === CUSTOMIZE HERE: Select which algorithms to test ===
# Options: Algorithms.GREEDY, Algorithms.ITAI, Algorithms.LUBY
SELECTED_ALGORITHMS = [Algorithms.GREEDY, Algorithms.ITAI, Algorithms.LUBY]
# Example: Test only Greedy and Luby
# SELECTED_ALGORITHMS = [Algorithms.GREEDY, Algorithms.LUBY]

# GA parameters
GA_CONFIG = GAConfig(
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    mutation_rate=MUTATION_RATE,
    use_cascading=False
)

# Cascading GA (optional)
GA_CONFIG_CASCADING = GAConfig(
    population_size=POPULATION_SIZE,
    generations=GENERATIONS,
    mutation_rate=MUTATION_RATE,
    use_cascading=True
)

# Create MetaConfig with selected algorithms
config_standard = MetaConfig(algorithms=SELECTED_ALGORITHMS, ga_config=GA_CONFIG)
config_cascading = MetaConfig(algorithms=SELECTED_ALGORITHMS, ga_config=GA_CONFIG_CASCADING)

algo_names_str = ', '.join([a.value for a in SELECTED_ALGORITHMS])
print(f'Configuration:')
print(f'  Graph: {NR_OF_NODES} nodes')
print(f'  Seeds: {SEEDS}')
print(f'  ✓ Algorithms: {algo_names_str}')
print(f'  GA: pop={GA_CONFIG.population_size}, gen={GA_CONFIG.generations}')
print(f'  Registry: {AlgorithmRegistry.instance()}')
print()

Configuration:
  Graph: 1000 nodes
  Seeds: [43, 73, 98]
  ✓ Algorithms: greedy, itai, luby
  GA: pop=20, gen=10
  Registry: AlgorithmRegistry(algorithms=['greedy', 'itai', 'luby'])



## 4. Algorithm Configuration Details

In [14]:
# Display selected algorithms and their parameters
print('='*100)
print('SELECTED ALGORITHMS & PARAMETER RANGES')
print('='*100 + '\n')

algo_names = [a.value for a in SELECTED_ALGORITHMS]
registry = AlgorithmRegistry.instance()

for algo_name in algo_names:
    algo_def = registry.get(algo_name)
    if algo_def:
        print(f'\n{algo_name.upper()} Algorithm')
        print(f'  Parameters:')
        for param_name, (min_val, max_val, _) in algo_def['parameters'].items():
            if isinstance(min_val, float):
                print(f'    {param_name:.<35} [{min_val:.1f}, {max_val:.1f}]')
            else:
                print(f'    {param_name:.<35} [{min_val}, {max_val}]')

# Show base parameters (used by cascading evaluator)
print(f'\n\nBASE PARAMETERS (System-Level)')
print(f'  max_iterations:...............................[5, 100]  (rounds of algorithm execution)')
print(f'  convergence_threshold:........................[0.0, 0.1] (stop if improvement < threshold)')

print('\n' + '='*100 + '\n')

SELECTED ALGORITHMS & PARAMETER RANGES


GREEDY Algorithm
  Parameters:
    max_rounds......................... [5, 100]

ITAI Algorithm
  Parameters:
    timeout_rounds..................... [1, 20]
    max_rounds......................... [5, 100]

LUBY Algorithm
  Parameters:
    base_probability................... [0.0, 1.0]
    coeff_degree....................... [-1.0, 1.0]
    coeff_neighbors_unmatched.......... [-1.0, 1.0]
    coeff_clustering................... [-1.0, 1.0]
    coeff_matched...................... [-1.0, 1.0]
    coeff_round........................ [-1.0, 1.0]
    coeff_weight....................... [-1.0, 1.0]
    max_rounds......................... [5, 100]


BASE PARAMETERS (System-Level)
  max_iterations:...............................[5, 100]  (rounds of algorithm execution)
  convergence_threshold:........................[0.0, 0.1] (stop if improvement < threshold)




## 5. Main Execution - Dynamic GA Evaluation

In [ ]:
# Run GA optimization across all seeds
all_results = {}
registry = AlgorithmRegistry.instance()

for seed in SEEDS:
    print('\n' + '='*80)
    print(f'SEED {seed}')
    print('='*80 + '\n')

    # Load graph
    print(f'Loading {NR_OF_NODES}-node graph (Seed: {seed})...', end=' ', flush=True)
    start_load = time.time()
    fixture = _create_clustered_graph(nr_of_nudes=NR_OF_NODES, seed=seed)
    graph = fixture_to_graph(fixture)
    print(f'✓ ({format_time(time.time() - start_load)})')

    # Compute optimal (NetworkX)
    print(f'Computing optimal weight (NetworkX)...', end=' ', flush=True)
    start = time.time()
    optimal = get_optimal_weight(fixture)
    print(f'✓ {optimal:.0f} ({format_time(time.time() - start)})')

    # Baselines
    print(f'Computing baseline (standard)...', end=' ', flush=True)
    start = time.time()
    baseline = get_baseline_fitness(graph, config_standard)
    print(f'✓ {baseline:.0f} ({format_time(time.time() - start)})')

    print(f'Computing baseline (cascading)...', end=' ', flush=True)
    start = time.time()
    baseline_cascading = get_cascading_baseline(graph, config_cascading)
    print(f'✓ {baseline_cascading:.0f} ({format_time(time.time() - start)})')

    # Individual algorithms
    print(f'Computing individual algorithm weights...', end=' ', flush=True)
    start = time.time()
    algo_weights = get_individual_algorithm_weights(graph, SELECTED_ALGORITHMS)
    print(f'✓ ({format_time(time.time() - start)})')
    algo_str = ', '.join([f'{a}: {algo_weights[a]:.0f}' for a in algo_weights])
    print(f'  {algo_str}\n')

    # GA Standard
    print(f'Running GA Standard with {algo_str}...')
    start = time.time()
    best_vector_standard, fitness_history_standard = run_ga_evaluation(graph, config_standard)
    best_standard = fitness_history_standard[-1]
    time_ga_std = time.time() - start
    print(f'✓ Done in {format_time(time_ga_std)}\n')

    # GA Cascading
    print(f'Running GA Cascading with {algo_str}...')
    start = time.time()
    best_vector_cascading, fitness_history_cascading = run_ga_evaluation(graph, config_cascading)
    best_cascading = fitness_history_cascading[-1]
    time_ga_casc = time.time() - start
    print(f'✓ Done in {format_time(time_ga_casc)}\n')

    # Results
    print('='*80)
    print('RESULTS')
    print('='*80)
    for algo, weight in algo_weights.items():
        print(f'{algo.upper():30} {weight:>12.0f}')
    print()
    print(f'Baseline (standard):         {baseline:>12.0f}')
    print(f'Baseline (cascading):        {baseline_cascading:>12.0f}')
    print()
    print(f'GA Standard:                 {best_standard:>12.0f}')
    print(f'GA Cascading:                {best_cascading:>12.0f}')
    print()
    print(f'NetworkX Optimal:            {optimal:>12.0f}')
    print()

    # Metrics
    improvement = ((best_standard - baseline) / (baseline + 1e-10)) * 100
    gap = ((optimal - best_standard) / (optimal + 1e-10)) * 100

    print('='*80)
    print('METRICS')
    print('='*80)
    print(f'GA Improvement (vs Baseline): {improvement:>11.2f}%')
    print(f'Gap to Optimal:              {gap:>11.2f}%')
    print()

    # Store results - IMPORTANT: Store BOTH vector objects AND fitness values
    all_results[seed] = {
        'optimal': optimal,
        'baseline': baseline,
        'baseline_cascading': baseline_cascading,
        'algo_weights': algo_weights,
        'best_vector_standard': best_vector_standard,
        'best_vector_cascading': best_vector_cascading,
        'best_standard': best_standard,
        'best_cascading': best_cascading,
        'fitness_history_standard': fitness_history_standard,
        'fitness_history_cascading': fitness_history_cascading,
        'improvement': improvement,
        'gap': gap,
    }

print('\n' + '='*80)
print('SUMMARY ACROSS ALL SEEDS')
print('='*80 + '\n')

print(f'{"Seed":<8} {"Baseline":<12} {"GA Std":<12} {"GA Casc":<12} {"Improvement":<12} {"Gap":<10}')
print('-' * 80)

improvements = []
gaps = []

for seed in SEEDS:
    r = all_results[seed]
    improvements.append(r['improvement'])
    gaps.append(r['gap'])
    print(f'{seed:<8} {r["baseline"]:>11.0f} {r["best_standard"]:>11.0f} {r["best_cascading"]:>11.0f} {r["improvement"]:>11.2f}% {r["gap"]:>9.2f}%')

print('-' * 80)
avg_improvement = sum(improvements) / len(improvements) if improvements else 0
avg_gap = sum(gaps) / len(gaps) if gaps else 0
print(f'{"AVG":<8} {" ":>11} {" ":>11} {" ":>11} {avg_improvement:>11.2f}% {avg_gap:>9.2f}%')
print('='*80)
print()
print(f'✓ GA evaluation complete with algorithms: {algo_names_str}')


SEED 43

Loading 1000-node graph (Seed: 43)... ✓ (0.0s)
Computing optimal weight (NetworkX)... ✓ 45327 (1.3s)
Computing baseline (standard)... ✓ 39131 (4.5s)
Computing baseline (cascading)... ✓ 40232 (7.3s)
Computing individual algorithm weights... ✓ (4.4s)
  greedy: 38643, itai: 34931, luby: 22616

Running GA Standard with greedy: 38643, itai: 34931, luby: 22616...
✓ Done in 15.7m

Running GA Cascading with greedy: 38643, itai: 34931, luby: 22616...
✓ Done in 25.8m

RESULTS
GREEDY                                38643
ITAI                                  34931
LUBY                                  22616

Baseline (standard):                39131
Baseline (cascading):               40232

GA Standard:                        39863
GA Cascading:                       40900

NetworkX Optimal:                   45327

METRICS
GA Improvement (vs Baseline):        1.87%
Gap to Optimal:                    12.05%


SEED 73

Loading 1000-node graph (Seed: 73)... ✓ (0.0s)
Computing optimal weig

## 6. Analysis - Generation-by-Generation Breakdown (Dynamic)

In [ ]:
# Dynamic analysis for each seed (shows generation-by-generation breakdown for all algorithms)
print('\n' + '='*100)
print('GENERATION-BY-GENERATION ANALYSIS (PER SEED)')
print('='*100 + '\n')

for seed in SEEDS:
    r = all_results[seed]
    fitness_history = r['fitness_history_standard']
    optimal = r['optimal']
    baseline = r['baseline']
    
    print(f'SEED {seed}')
    print('-'*100)
    print(f'{"Gen":>3} {"Fitness":>12} {"Change":>12} {"% Change":>12} {"Gap to Opt":>12}')
    print('-'*100)
    
    improvements = [0.0] + [fitness_history[i] - fitness_history[i-1] for i in range(1, len(fitness_history))]
    
    for gen, fitness in enumerate(fitness_history, 1):
        change = improvements[gen-1]
        pct_change = (change / (baseline + 1e-10)) * 100
        gap_to_opt = ((optimal - fitness) / (optimal + 1e-10)) * 100
        print(f"{gen:>3} {fitness:>12.0f} {change:>12.0f} {pct_change:>11.2f}% {gap_to_opt:>11.2f}%")
    
    print('-'*100)
    total_improvement = fitness_history[-1] - fitness_history[0]
    total_pct = (total_improvement / (baseline + 1e-10)) * 100
    print(f"Total improvement: {total_improvement:.0f} ({total_pct:+.2f}%)")
    print()

## 7. Analysis - Parameter Evolution

In [ ]:
# Analyze parameter values evolved by GA across all seeds
print('='*100)
print('PARAMETER ANALYSIS - VALUES EVOLVED BY GA')
print('='*100 + '\n')

# Collect parameter values from best vectors
best_vectors = {seed: all_results[seed]['best_vector_standard'] for seed in SEEDS}
param_values = {}  # param_name -> [values from each seed]

# Extract values from best vectors (CanonicalVector has fixed parameters)
param_names = [
    'luby_base_probability',
    'luby_coeff_degree', 
    'luby_coeff_neighbors_unmatched',
    'luby_coeff_clustering',
    'luby_coeff_matched',
    'luby_coeff_round',
    'luby_coeff_weight',
    'itai_timeout_rounds',
    'max_iterations',
    'convergence_threshold'
]

for seed, vector in best_vectors.items():
    for param_name in param_names:
        if hasattr(vector, param_name):
            if param_name not in param_values:
                param_values[param_name] = []
            param_values[param_name].append(getattr(vector, param_name))

# Display analysis
print(f'{"Parameter":<40} {"Min":<12} {"Max":<12} {"Mean":<12} {"Range"}')
print('-'*100)

for param_name in sorted(param_values.keys()):
    values = param_values[param_name]
    if values:
        min_val = min(values)
        max_val = max(values)
        mean_val = sum(values) / len(values)
        range_val = max_val - min_val
        
        # Format based on type
        if isinstance(min_val, float):
            print(f'{param_name:<40} {min_val:<12.4f} {max_val:<12.4f} {mean_val:<12.4f} {range_val:.4f}')
        else:
            print(f'{param_name:<40} {min_val:<12} {max_val:<12} {mean_val:<12.1f} {range_val}')

print('\n' + '='*100 + '\n')

## 8. Visualization - Fitness Progression (Dynamic)

In [ ]:
# Plot fitness progression for each seed (Standard vs Cascading) + individual algorithm baselines
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, seed in enumerate(SEEDS):
    r = all_results[seed]
    fitness_history_std = r['fitness_history_standard']
    fitness_history_casc = r['fitness_history_cascading']
    baseline = r['baseline']
    baseline_cascading = r['baseline_cascading']
    optimal = r['optimal']
    algo_weights = r['algo_weights']
    
    ax = axes[idx]
    gens = range(1, len(fitness_history_std) + 1)
    
    # Plot GA fitness progression
    ax.plot(gens, fitness_history_std, 'o-', linewidth=2.5, markersize=7, label='GA Standard', color='blue')
    ax.plot(gens, fitness_history_casc, 's-', linewidth=2.5, markersize=7, label='GA Cascading', color='green')
    
    # Plot merged baselines
    ax.axhline(baseline, linestyle='--', color='orange', linewidth=2, label=f'Merged Baseline (Std): {baseline:.0f}')
    ax.axhline(baseline_cascading, linestyle='--', color='purple', linewidth=2, label=f'Merged Baseline (Casc): {baseline_cascading:.0f}')
    
    # Plot individual algorithm baselines
    colors_algo = {'greedy': '#d62728', 'itai': '#ff7f0e', 'luby': '#2ca02c'}
    for algo_name, weight in algo_weights.items():
        color = colors_algo.get(algo_name, '#1f77b4')
        ax.axhline(weight, linestyle=':', color=color, linewidth=1.5, alpha=0.7, label=f'{algo_name.upper()}: {weight:.0f}')
    
    # Plot optimal
    ax.axhline(optimal, linestyle='-', color='red', linewidth=2.5, label=f'Optimal: {optimal:.0f}')
    
    ax.set_xlabel('Generation', fontsize=11, fontweight='bold')
    ax.set_ylabel('Fitness', fontsize=11, fontweight='bold')
    ax.set_title(f'Seed {seed}', fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='lower right')
    ax.set_ylim(min(fitness_history_std + list(algo_weights.values())) * 0.95, optimal * 1.05)

plt.suptitle(f'GA Fitness Progression with Algorithm Baselines - Algorithms: {algo_names_str}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('✓ Fitness progression plot with individual algorithm baselines complete')

## 9. Visualization - Baseline Comparison (Dynamic)

In [ ]:
# Compare individual algorithms + merged baseline vs GA results + optimal
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for idx, seed in enumerate(SEEDS):
    ax = axes[idx]
    r = all_results[seed]
    
    # Prepare data: individual algorithms + merged baselines + GA results + optimal
    algo_weights = r['algo_weights']
    baseline = r['baseline']
    baseline_cascading = r['baseline_cascading']
    optimal = r['optimal']
    best_standard = r['best_standard']
    best_cascading = r['best_cascading']
    
    # Create labels and values
    # Individual algorithms first, then merged, then GA, then optimal
    labels = (
        list(algo_weights.keys()) + 
        ['Merged\n(Std)', 'Merged\n(Casc)', 'GA\n(Std)', 'GA\n(Casc)', 'Optimal']
    )
    values = (
        list(algo_weights.values()) + 
        [baseline, baseline_cascading, best_standard, best_cascading, optimal]
    )
    
    # Color scheme
    algo_colors = ['#1f77b4', '#ff7f0e', '#2ca02c'][:len(algo_weights)]  # Blue, Orange, Green for algorithms
    merged_colors = ['#d62728', '#9467bd']  # Red, Purple for merged baselines
    ga_colors = ['#8c564b', '#e377c2']  # Brown, Pink for GA
    optimal_color = ['#bcbd22']  # Yellow-green for optimal
    colors = algo_colors + merged_colors + ga_colors + optimal_color
    
    # Create bars
    bars = ax.bar(labels, values, color=colors, alpha=0.75, edgecolor='black', linewidth=1.5)
    
    # Add value labels on bars
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.0f}',
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    
    ax.set_ylabel('Weight', fontsize=11, fontweight='bold')
    ax.set_title(f'Seed {seed}', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    ax.set_ylim(0, optimal * 1.15)
    
    # Rotate x labels for readability
    ax.tick_params(axis='x', rotation=45)

plt.suptitle(f'Baseline Comparison: Individual Algorithms + Merged + GA - {algo_names_str}', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n" + "="*100)
print("BASELINE COMPARISON - Individual Algorithm Contributions")
print("="*100 + "\n")
for seed in SEEDS:
    r = all_results[seed]
    print(f'Seed {seed}:')
    for algo_name, weight in r['algo_weights'].items():
        print(f'  {algo_name.upper():20} {weight:>10.0f}')
    print(f'  {"-" * 30}')
    print(f'  {"Merged (Standard)":20} {r["baseline"]:>10.0f}')
    print(f'  {"Merged (Cascading)":20} {r["baseline_cascading"]:>10.0f}')
    print(f'  {"-" * 30}')
    print(f'  {"GA (Standard)":20} {r["best_standard"]:>10.0f}')
    print(f'  {"GA (Cascading)":20} {r["best_cascading"]:>10.0f}')
    print(f'  {"-" * 30}')
    print(f'  {"Optimal (NetworkX)":20} {r["optimal"]:>10.0f}')
    print()

## 10. Visualization - Performance Metrics (Dynamic)

In [ ]:
# Compare improvements and gaps to optimal across approaches
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Improvement comparison
ax = axes[0]
x_pos = np.arange(len(SEEDS))
width = 0.25

improvements_std = [all_results[seed]['improvement'] for seed in SEEDS]
gaps = [all_results[seed]['gap'] for seed in SEEDS]

bars1 = ax.bar(x_pos - width, improvements_std, width, label='GA Improvement (%)', color='blue', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x_pos, gaps, width, label='Gap to Optimal (%)', color='red', alpha=0.7, edgecolor='black')
bars3 = ax.bar(x_pos + width, [1.0] * len(SEEDS), width, label='Target (1% gap)', color='green', alpha=0.3, edgecolor='black')

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylabel('Percentage (%)', fontsize=11, fontweight='bold')
ax.set_title('GA Improvement vs Gap to Optimal', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Seed {s}' for s in SEEDS], fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Plot 2: Fitness comparison (all approaches)
ax = axes[1]
baselines = [all_results[seed]['baseline'] for seed in SEEDS]
gas_std = [all_results[seed]['best_standard'] for seed in SEEDS]
gas_casc = [all_results[seed]['best_cascading'] for seed in SEEDS]
optima = [all_results[seed]['optimal'] for seed in SEEDS]

x_pos = np.arange(len(SEEDS))
width = 0.2

bars1 = ax.bar(x_pos - 1.5*width, baselines, width, label='Baseline', color='orange', alpha=0.7, edgecolor='black')
bars2 = ax.bar(x_pos - 0.5*width, gas_std, width, label='GA Standard', color='blue', alpha=0.7, edgecolor='black')
bars3 = ax.bar(x_pos + 0.5*width, gas_casc, width, label='GA Cascading', color='green', alpha=0.7, edgecolor='black')
bars4 = ax.bar(x_pos + 1.5*width, optima, width, label='Optimal', color='red', alpha=0.7, edgecolor='black')

ax.set_ylabel('Fitness (Weight)', fontsize=11, fontweight='bold')
ax.set_title('Fitness Comparison Across All Approaches', fontsize=12, fontweight='bold')
ax.set_xticks(x_pos)
ax.set_xticklabels([f'Seed {s}' for s in SEEDS], fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('✓ Performance metrics plot complete')

## 11. Visualization - Parameter Space Exploration

In [ ]:
# Visualize parameter evolution: show bounds vs actual values found
algo_names = [a.value for a in SELECTED_ALGORITHMS]

# Parameter bounds (from CanonicalVector)
param_bounds = {
    'luby_base_probability': (0.0, 1.0),
    'luby_coeff_degree': (-1.0, 1.0),
    'luby_coeff_neighbors_unmatched': (-1.0, 1.0),
    'luby_coeff_clustering': (-1.0, 1.0),
    'luby_coeff_matched': (-1.0, 1.0),
    'luby_coeff_round': (-1.0, 1.0),
    'luby_coeff_weight': (-1.0, 1.0),
    'itai_timeout_rounds': (1, 20),
    'max_iterations': (5, 100),
    'convergence_threshold': (0.0, 0.1),
}

# Get best vector values
best_vector = all_results[SEEDS[0]]['best_vector_standard']
param_names = [p for p in param_bounds.keys() if hasattr(best_vector, p)]

# Separate float and int parameters for better visualization
float_params = []
int_params = []

for param_name in param_names:
    min_val, max_val = param_bounds[param_name]
    if isinstance(min_val, float):
        float_params.append(param_name)
    else:
        int_params.append(param_name)

# Create separate plots for float and int parameters
if float_params:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x_pos = np.arange(len(float_params))
    
    for i, param_name in enumerate(float_params):
        min_val, max_val = param_bounds[param_name]
        
        # Get all values from all seeds
        all_values = []
        for seed in SEEDS:
            vector = all_results[seed]['best_vector_standard']
            if hasattr(vector, param_name):
                all_values.append(getattr(vector, param_name))
        
        if all_values:
            mean_val = np.mean(all_values)
            
            # Draw range
            ax.plot([i, i], [min_val, max_val], 'k-', linewidth=3, alpha=0.3, label='Search Range' if i == 0 else '')
            
            # Draw all found values
            for val in all_values:
                ax.scatter(i, val, s=150, marker='o', color='blue', alpha=0.6, edgecolors='darkblue', linewidth=2)
            
            # Draw mean
            ax.scatter(i, mean_val, s=250, marker='*', color='red', edgecolors='darkred', linewidth=2, label='Mean' if i == 0 else '', zorder=5)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(float_params, rotation=45, ha='right', fontweight='bold')
    ax.set_ylabel('Parameter Value', fontsize=11, fontweight='bold')
    ax.set_title(f'Float Parameters: Search Range vs Evolved Values', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

if int_params:
    fig, ax = plt.subplots(figsize=(14, 6))
    
    x_pos = np.arange(len(int_params))
    
    for i, param_name in enumerate(int_params):
        min_val, max_val = param_bounds[param_name]
        
        # Get all values from all seeds
        all_values = []
        for seed in SEEDS:
            vector = all_results[seed]['best_vector_standard']
            if hasattr(vector, param_name):
                all_values.append(int(getattr(vector, param_name)))
        
        if all_values:
            mean_val = np.mean(all_values)
            
            # Draw range
            ax.plot([i, i], [min_val, max_val], 'k-', linewidth=3, alpha=0.3, label='Search Range' if i == 0 else '')
            
            # Draw all found values
            for val in all_values:
                ax.scatter(i, val, s=150, marker='o', color='green', alpha=0.6, edgecolors='darkgreen', linewidth=2)
            
            # Draw mean
            ax.scatter(i, mean_val, s=250, marker='*', color='red', edgecolors='darkred', linewidth=2, label='Mean' if i == 0 else '', zorder=5)
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(int_params, rotation=45, ha='right', fontweight='bold')
    ax.set_ylabel('Parameter Value', fontsize=11, fontweight='bold')
    ax.set_title(f'Integer Parameters: Search Range vs Evolved Values', fontsize=12, fontweight='bold')
    ax.legend(fontsize=10, loc='upper right')
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.show()

print('✓ Parameter space visualization complete')

In [ ]:
# Inspect actual matched edges from algorithms - ALL SEEDS
print('='*100)
print('MATCHED EDGES - Inspect Actual Matches from Algorithms (ALL SEEDS)')
print('='*100 + '\n')

# Loop through all seeds
for seed_inspect in SEEDS:
    print('\n' + '='*100)
    print(f'SEED {seed_inspect}')
    print('='*100 + '\n')
    
    # Reload graph for this seed
    fixture = _create_clustered_graph(nr_of_nudes=NR_OF_NODES, seed=seed_inspect)
    graph = fixture_to_graph(fixture)
    
    # Get the best vector from GA
    best_vector = all_results[seed_inspect]['best_vector_standard']
    
    print(f'Best GA Vector Parameters:')
    param_names = [
        'luby_base_probability',
        'luby_coeff_degree', 
        'luby_coeff_neighbors_unmatched',
        'luby_coeff_clustering',
        'luby_coeff_matched',
        'luby_coeff_round',
        'luby_coeff_weight',
        'itai_timeout_rounds',
        'max_iterations',
        'convergence_threshold'
    ]
    for param_name in param_names:
        if hasattr(best_vector, param_name):
            value = getattr(best_vector, param_name)
            if isinstance(value, float):
                print(f'  {param_name:.<40} {value:.4f}')
            else:
                print(f'  {param_name:.<40} {value}')
    
    print('\n' + '-'*100 + '\n')
    
    # Run ONLY SELECTED algorithms to see their matches
    algo_names_list = [a.value for a in SELECTED_ALGORITHMS]
    
    for algo_name in algo_names_list:
        print(f'{algo_name.upper()} Algorithm Matching')
        print('-'*100)
        
        # Create parameterizer for this algorithm
        param = UnifiedAlgorithmParameterizer(algo_name)
        
        # Execute to get matching
        matching = param.execute(graph, best_vector)
        
        # Calculate weight
        total_weight = sum(graph.get_edge_weight(u, v) for u, v in matching.items() if u < v)
        
        print(f'  Total edges matched: {len(matching) // 2}  (dict size: {len(matching)})')
        print(f'  Total weight: {total_weight:.0f}')
        print(f'\n  Edge list (first 10):')
        
        edges = [(u, v, graph.get_edge_weight(u, v)) for u, v in matching.items() if u < v]
        edges.sort(key=lambda x: -x[2])  # Sort by weight descending
        
        for i, (u, v, w) in enumerate(edges[:10]):
            print(f'    {i+1:2d}. ({u:3d}, {v:3d}) weight: {w:>7.2f}')
        
        if len(edges) > 10:
            print(f'    ... and {len(edges) - 10} more edges')
        
        print()
    
    print('-'*100)
    print(f'MERGED RESULT - SELECTED Algorithms Combined (with conflict resolution)')
    print(f'Selected: {", ".join(algo_names_list)}')
    print('-'*100 + '\n')
    
    # Run ONLY selected algorithms and merge
    matchings = []
    algo_names_list = [a.value for a in SELECTED_ALGORITHMS]
    
    for algo_name in algo_names_list:
        param = UnifiedAlgorithmParameterizer(algo_name)
        matching = param.execute(graph, best_vector)
        matchings.append(matching)
    
    # Import merge function from correct location
    from src.meta.core.matching_merger import merge_matchings
    
    merged = merge_matchings(matchings, graph)
    total_weight = sum(graph.get_edge_weight(u, v) for u, v in merged.items() if u < v)
    
    print(f'Total edges in merged result: {len(merged) // 2}')
    print(f'Total weight: {total_weight:.0f}')
    print(f'\nEdges (sorted by weight, descending):')
    
    edges = [(u, v, graph.get_edge_weight(u, v)) for u, v in merged.items() if u < v]
    edges.sort(key=lambda x: -x[2])
    
    for i, (u, v, w) in enumerate(edges[:20]):
        print(f'  {i+1:2d}. ({u:3d}, {v:3d}) weight: {w:>7.2f}')
    
    if len(edges) > 20:
        print(f'  ... and {len(edges) - 20} more edges')

print('\n' + '='*100)
print('END OF INSPECTION - ALL SEEDS COMPLETE')
print('='*100)

## 12. Inspection - View Actual Matched Edges